# CO2BLOCK-informed resource-ceiling filter on global 2050 storage rate

For each `(scenario, country, model)` we apply a CO2BLOCK-informed
resource ceiling to the Part-1 Monte-Carlo sample pool and rerun the
same bootstrap convolution used in `04_global_aggregate_2050` to
quantify how the global 2050 storage rate distribution shifts under
the geological constraint.

## Ceiling rule

For failed Part-2 cases (reconstructed in Part-3), the main
(conservative) ceiling is `C_cap_min = min(C_recon_order_gt)` over the
ascending/descending CO2BLOCK orderings, and the sensitivity ceiling is
`C_cap_mean = mean(C_recon_order_gt)` over orderings. For passed Part-2
cases, we use the scenario's design `C_scenario_gt` from Part-3
`part3_input_table.csv` (geology has already validated this).

## Filter and bootstrap

For each country's MC pool we keep only samples with
`C_sampled ≤ C_screened`, then bootstrap-convolve the surviving
samples across countries (N = 10 000 world draws / (scenario, model)).
We report sample counts before/after filtering and flag any country
pool that becomes empty under the conservative ceiling.

## What this filter is

This is a CO2BLOCK-informed resource-ceiling filter, not a full CO2BLOCK
run over every MC trajectory. CO2BLOCK is evaluated only on each
scenario's deterministic central trajectory in Part-2; the resulting
screened resource ceiling is then used to mask the Part-1 MC sample
pool. A future iteration could push CO2BLOCK through every trajectory.

## Outputs

* `output/global_2050_rate_C_ceiling_screening_bootstrap.csv`: full
  bootstrap rows (`scenario`, `model`, `ceiling_kind`, `unscreened`,
  `screened`).
* `output/C_ceiling_diagnostic_per_country_scenario_model.csv`:
  before/after sample counts, `C_cap`, `failed_Part2` flag.
* `output/C_ceiling_empty_pools.csv`: country cells whose filtered
  pool is empty under the conservative ceiling.
* `output/figures/fig_global_2050_rate_screening_impact.png`: main
  ridgeline figure.
* `output/figures/fig_C_ceiling_sensitivity_min_vs_mean.png`: small
  side-by-side comparison of min vs mean ceiling reductions.


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

os.environ.setdefault("MPLCONFIGDIR", '/private/tmp/mplconfig_iman2026')

# Locate repo root
for cand in [Path.cwd().resolve(), Path.cwd().resolve().parent,
             Path.cwd().resolve().parent.parent,
             Path('/Users/xg320/Desktop/IC-Sam-works/paper/Iman-2026')]:
    if (cand / '01_growth_model').exists():
        ROOT = cand; break
else:
    raise FileNotFoundError('repo root with 01_growth_model/ not found')

P1   = ROOT / '01_growth_model' / 'output' / 'v8_2026-06-01'
P3   = ROOT / '03_feasible_growth' / 'output'
OUT  = ROOT / '04_global_aggregate_2050' / 'output'
FIG  = OUT / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

# Config (shared with 04_global_aggregate_2050)
SCENARIOS = ['minimum', 'reference', 'growth10', 'policy',
             'us1gt', 'maximum', 'ipcc_high', 'ipcc_low']
MODELS    = ['Logistic', 'Gompertz']
N_BOOT    = 10_000
SEED      = 42

SCENARIO_COLOR = {
    # Paul Tol "Muted" palette, colourblind-safe, print-safe (Tol 2018).
    'minimum':   '#888888',  # medium grey (neutral baseline, readable)
    'growth10':  '#DDCC77',  # sand
    'ipcc_low':  '#88CCEE',  # light blue
    'policy':    '#44AA99',  # teal
    'us1gt':     '#117733',  # green
    'reference': '#332288',  # indigo
    'ipcc_high': '#AA4499',  # purple
    'maximum':   '#CC6677',  # rose
}
SCENARIO_LABEL = {
    'minimum':'Minimum', 'reference':'Reference', 'growth10':'Growth 10%',
    'policy':'Policy', 'us1gt':'US 1 Gt', 'maximum':'Maximum',
    'ipcc_high':'IPCC High', 'ipcc_low':'IPCC Low',
}
MODEL_STYLE = {'Logistic': {'ls':'-',  'lw':1.7, 'fill_alpha':0.28},
               'Gompertz': {'ls':'--', 'lw':1.7, 'fill_alpha':0.16}}

plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
})
print('repo :', ROOT)
print('out  :', OUT)


In [ ]:
# 1. Build C ceilings per (scenario, country, model)
# Only the SCENARIO-SCREENED C is used:
#   * failed Part-2 -> C_recon_order_gt  (min/mean across orderings)
#   * passed Part-2 -> C_scenario_gt     (scenario design value)
# Sources are Part-3 outputs (v8):
#   * part3_input_table.csv             -> C_scenario_gt for ALL 150 central
#     (scenario,country,model) cells. scalars_central.csv is NOT used: it
#     omits cells (only the deterministic scalar rows) and would silently
#     drop model-paths.
#   * smooth_reconstruction_summary.csv -> C_recon_order_gt for the failed
#     (scenario,country,model,ordering) reconstruction targets.
# No additional country geological-capacity floor is applied: CO2BLOCK
# already enters via the screening pass/fail and the reconstructed C for
# failed cases. Note that for every v8 MC sample C_sampled <= C_scenario_gt
# (0 violations / 316,029 samples), so the passed-case ceiling is a no-op
# and ALL geological reduction comes from the failed cells' C_recon.
pit       = pd.read_csv(P3 / 'part3_input_table.csv')
recon_sum = pd.read_csv(P3 / 'smooth_reconstruction_summary.csv')

cap_min  = (recon_sum.groupby(['scenario','country','model'])
                      ['C_recon_order_gt'].min().to_dict())
cap_mean = (recon_sum.groupby(['scenario','country','model'])
                      ['C_recon_order_gt'].mean().to_dict())
# C_scenario_gt is identical across orderings/models for a (scenario,country);
# key by (scenario, country) to match the diagnostic built downstream.
C_scenario_lookup = {(r.scenario, r.country): float(r.C_scenario_gt)
                      for r in pit.itertuples()}
failed_set = set(cap_min.keys())

def get_cap(scen, country, mdl, kind):
    key = (scen, country, mdl)
    if key in failed_set:
        return float((cap_min if kind == 'min' else cap_mean)[key])
    return C_scenario_lookup[(scen, country)]

# Strict check: every actual MC sample cell must resolve a C cap
mc_cells = set()
for scen in SCENARIOS:
    sp = P1 / f'samples_{scen}.csv'
    if not sp.exists():
        continue
    sdf = pd.read_csv(sp, usecols=['Country','Model'])
    for c in sdf['Country'].unique():
        for mdl in sdf.loc[sdf['Country'] == c, 'Model'].unique():
            mc_cells.add((scen, c, mdl))
missing_cap = [cell for cell in sorted(mc_cells)
               if (cell not in failed_set)
               and ((cell[0], cell[1]) not in C_scenario_lookup)]
assert not missing_cap, f'MC cells with NO resolvable C cap: {missing_cap}'
print(f'MC cells (scenario,country,model): {len(mc_cells)}  - all resolve a C ceiling')

n_central = pit[['scenario','country','model']].drop_duplicates().shape[0]
print(f'Failed Part-2 cells with reconstruction: {len(failed_set)}')
print(f'Passed (use C_scenario_gt):               '
      f'{n_central - len(failed_set)}  /  {n_central} central cells')

# Save the resolved cap table (all central cells, no basin floor)
cap_rows = []
for cell in pit[['scenario','country','model']].drop_duplicates().itertuples(index=False):
    scen, c, mdl = cell.scenario, cell.country, cell.model
    key = (scen, c, mdl)
    failed = key in failed_set
    cap_rows.append({
        'scenario':scen, 'country':c, 'model':mdl,
        'failed_Part2':failed,
        'C_scenario_gt':C_scenario_lookup[(scen, c)],
        'C_recon_min_gt':float(cap_min[key])  if failed else np.nan,
        'C_recon_mean_gt':float(cap_mean[key]) if failed else np.nan,
        'C_cap_final_min':  get_cap(scen, c, mdl, 'min'),
        'C_cap_final_mean': get_cap(scen, c, mdl, 'mean'),
    })
caps_df = pd.DataFrame(cap_rows)
caps_df.to_csv(OUT / 'C_ceiling_resolved_per_cell.csv', index=False)
print(f'\nSaved: {OUT / "C_ceiling_resolved_per_cell.csv"}  ({len(caps_df)} cells)')

In [ ]:
# 2. Filter MC pool by C_sampled <= C_screened, then bootstrap
def filter_and_bootstrap(kind, seed_offset):
    # For each (scenario, model): filter every country's Part-1 MC pool
    # to {C_sampled <= ceiling}, then bootstrap-sum across countries.
    # Countries whose filtered pool is EMPTY are EXCLUDED from the
    # post-screening bootstrap (their contribution is effectively zero).
    # Returns (boot_df, diag_df, empty_df).
    rng = np.random.default_rng(SEED + seed_offset)
    boot_rows, diag_rows, empty_pools = [], [], []
    for scen in SCENARIOS:
        samp = pd.read_csv(P1 / f'samples_{scen}.csv')
        countries = sorted(samp['Country'].unique())
        for mdl in MODELS:
            sub = samp[samp['Model'] == mdl]
            un_arr, sc_arr = {}, {}
            for c in countries:
                csub = sub[sub['Country'] == c]
                if csub.empty:
                    continue
                r_un  = csub['rate_2050'].to_numpy()
                C_smp = csub['C_sampled'].to_numpy()
                cap   = get_cap(scen, c, mdl, kind)
                keep  = C_smp <= cap
                r_sc  = r_un[keep]
                diag_rows.append({
                    'scenario':scen, 'country':c, 'model':mdl,
                    'ceiling_kind':kind, 'C_cap_gt':cap,
                    'C_scenario_gt':C_scenario_lookup[(scen, c)],
                    'failed_Part2': (scen, c, mdl) in failed_set,
                    'n_samples_before': int(csub.shape[0]),
                    'n_samples_after':  int(keep.sum()),
                    'kept_frac': float(keep.mean()) if keep.size else 0.0,
                    'median_r2050_before': float(np.median(r_un)) if r_un.size else np.nan,
                    'median_r2050_after':  float(np.median(r_sc)) if r_sc.size else np.nan,
                    'excluded_from_post_screening_bootstrap':
                        (r_sc.size == 0),
                })
                un_arr[c] = r_un
                if r_sc.size == 0:
                    empty_pools.append({'scenario':scen,'country':c,'model':mdl,
                                         'ceiling_kind':kind,'C_cap_gt':cap,
                                         'min_C_sampled':float(C_smp.min()),
                                         'min_r2050':float(np.min(r_un))})
                    continue   # exclude this country from post-screening bootstrap
                sc_arr[c] = r_sc

            if not un_arr:
                continue
            w_un = np.zeros(N_BOOT)
            for c, a in un_arr.items():
                w_un += rng.choice(a, size=N_BOOT, replace=True)
            w_sc = np.zeros(N_BOOT)
            for c, a in sc_arr.items():
                w_sc += rng.choice(a, size=N_BOOT, replace=True)
            for u, s in zip(w_un, w_sc):
                boot_rows.append({'scenario':scen, 'model':mdl,
                                   'ceiling_kind':kind,
                                   'unscreened':float(u),
                                   'screened':float(s)})
    return (pd.DataFrame(boot_rows),
             pd.DataFrame(diag_rows),
             pd.DataFrame(empty_pools))

print('Running MAIN: min(C_recon_order_gt) ceiling ...')
main_df,  main_diag, main_empty  = filter_and_bootstrap('min',  3)
print('Running SENSITIVITY: mean(C_recon_order_gt) ceiling ...')
sens_df,  sens_diag, sens_empty  = filter_and_bootstrap('mean', 4)

boot_all = pd.concat([main_df, sens_df], ignore_index=True)
diag_all = pd.concat([main_diag, sens_diag], ignore_index=True)
empty_all = pd.concat([main_empty, sens_empty], ignore_index=True) if (len(main_empty)+len(sens_empty)) else pd.DataFrame()

boot_all.to_csv (OUT / 'global_2050_rate_C_ceiling_screening_bootstrap.csv', index=False)
diag_all.to_csv (OUT / 'C_ceiling_diagnostic_per_country_scenario_model.csv', index=False)
if len(empty_all):
    empty_all.to_csv(OUT / 'C_ceiling_empty_pools.csv', index=False)
print(f'\nbootstrap rows  : {len(boot_all):,}')
print(f'diagnostic rows : {len(diag_all):,}')
print(f'empty pool rows : {len(empty_all)}  (these countries are EXCLUDED from post-screening bootstrap)')


In [ ]:
# 3. Diagnostic reporting
print('=== Sample-count diagnostic (MAIN, min ceiling) ===')
agg_main = (main_diag.groupby('scenario')
              .agg(n_before=('n_samples_before','sum'),
                   n_after =('n_samples_after','sum'),
                   countries_in_scen=('country','nunique'))
              .reset_index())
agg_main['kept_pct'] = agg_main['n_after'] / agg_main['n_before'] * 100
print(agg_main.round(1).to_string(index=False))

print('\n=== Sample-count diagnostic (SENSITIVITY, mean ceiling) ===')
agg_sens = (sens_diag.groupby('scenario')
              .agg(n_before=('n_samples_before','sum'),
                   n_after =('n_samples_after','sum'))
              .reset_index())
agg_sens['kept_pct'] = agg_sens['n_after'] / agg_sens['n_before'] * 100
print(agg_sens.round(1).to_string(index=False))

print('\n=== Empty filtered pools ===')
if len(main_empty):
    print(f'MAIN (min ceiling) — {len(main_empty)} cell(s):')
    print(main_empty.round(3).to_string(index=False))
else:
    print('MAIN (min ceiling):  no empty pools.')
if len(sens_empty):
    print(f'\nSENSITIVITY (mean ceiling) — {len(sens_empty)} cell(s):')
    print(sens_empty.round(3).to_string(index=False))
else:
    print('SENSITIVITY (mean ceiling): no empty pools.')

# Median 2050 rate summary (unscreened, min, mean)
def med_tab(df, label):
    return (df.groupby(['scenario','model'])
              .agg(med_un=('unscreened','median'),
                   med_sc=('screened',  'median'))
              .reset_index()
              .rename(columns={'med_sc': f'med_{label}'}))
m_main = med_tab(main_df, 'min')
m_sens = med_tab(sens_df, 'mean')[['scenario','model','med_mean']]
summary = m_main.merge(m_sens, on=['scenario','model'])
summary['reduction_min_pct']  = (1 - summary['med_min']  / summary['med_un']) * 100
summary['reduction_mean_pct'] = (1 - summary['med_mean'] / summary['med_un']) * 100
print('\n=== Global 2050 rate median (Gt/yr) ===')
print(summary.round(2).to_string(index=False))
summary.to_csv(OUT / 'C_ceiling_2050_rate_median_summary.csv', index=False)


In [ ]:
# 6. Post-screening histogram (twin of fig_global_aggregate_2050_histograms panel)
# Same layout as the unscreened histogram in 04_global_aggregate_2050:
# single panel, 8 scenarios overlaid, per-scenario 32 bins, with the
# top dominance ribbon. The only change is the input: bootstrap rows
# now come from the CO2BLOCK-informed FILTERED sample pool.
from scipy.stats import gaussian_kde
from scipy.signal import medfilt

# Use the screened (min-ceiling) bootstrap as the input
screened_records = main_df.rename(columns={'screened':'rate_2050_gt_yr'})[
    ['scenario','model','rate_2050_gt_yr']].copy()

xmax_h = float(np.percentile(screened_records['rate_2050_gt_yr'], 99.5))

# Dominance ribbon: KDE over scenarios, median-filtered to drop artefacts
fine_x = np.linspace(0.01, xmax_h, 300)
dens   = np.zeros((len(SCENARIOS), len(fine_x)))
for i, scen in enumerate(SCENARIOS):
    v = screened_records[screened_records['scenario']==scen]['rate_2050_gt_yr'].to_numpy()
    if v.size > 1:
        dens[i] = gaussian_kde(v, bw_method=0.10)(fine_x)
dominant_idx = medfilt(np.argmax(dens, axis=0), kernel_size=11).astype(int)
runs = []
cur_i, cur_x0 = dominant_idx[0], fine_x[0]
for j in range(1, len(fine_x)):
    if dominant_idx[j] != cur_i:
        runs.append((cur_i, cur_x0, fine_x[j-1])); cur_i, cur_x0 = dominant_idx[j], fine_x[j]
runs.append((cur_i, cur_x0, fine_x[-1]))

fig, ax = plt.subplots(figsize=(13.5, 7.2))

label_anchor = {}
max_count_global = 0
for scen in SCENARIOS:
    col = SCENARIO_COLOR[scen]
    sub_scen = screened_records[screened_records['scenario'] == scen]
    if sub_scen.empty: continue
    s_lo = float(sub_scen['rate_2050_gt_yr'].min())
    s_hi = float(sub_scen['rate_2050_gt_yr'].max())
    pad  = max((s_hi - s_lo) * 0.04, 1e-3)
    lo, hi = max(0.0, s_lo - pad), s_hi + pad
    bins   = np.linspace(lo, hi, 33)        # 32 bins per scenario, own range
    bin_ctr = 0.5 * (bins[:-1] + bins[1:])
    peak_x = None
    for mdl in MODELS:
        v = sub_scen[sub_scen['model'] == mdl]['rate_2050_gt_yr'].to_numpy()
        if v.size == 0: continue
        counts, _ = np.histogram(v, bins=bins)
        if counts.max() == 0: continue
        max_count_global = max(max_count_global, int(counts.max()))
        sty = MODEL_STYLE[mdl]
        if sty['fill_alpha'] > 0:
            ax.fill_between(bin_ctr, counts, step='mid', color=col,
                              alpha=sty['fill_alpha'], linewidth=0)
        ax.stairs(counts, bins, color=col, linestyle=sty['ls'],
                   linewidth=sty['lw'], alpha=0.92, baseline=None)
        if mdl == 'Logistic':
            peak_x = float(bin_ctr[int(np.argmax(counts))])
    if peak_x is not None:
        label_anchor[scen] = peak_x

# staggered scenario labels at top with leader lines
y_lvls = [max_count_global * 1.08, max_count_global * 1.17, max_count_global * 1.26]
y_top  = max_count_global * 1.34
ordered = sorted(label_anchor.items(), key=lambda kv: kv[1])
for i, (scen, px) in enumerate(ordered):
    y_lab = y_lvls[i % len(y_lvls)]
    ax.plot([px, px],
             [max_count_global * 1.02, y_lab - max_count_global * 0.012],
             color=SCENARIO_COLOR[scen], lw=0.7, alpha=0.55, clip_on=False)
    ax.text(px, y_lab, SCENARIO_LABEL[scen],
             ha='center', va='bottom', fontsize=8.8, fontweight='bold',
             color=SCENARIO_COLOR[scen],
             bbox=dict(facecolor='white', edgecolor='none',
                       alpha=0.85, pad=1.2), clip_on=False)

# Dominance ribbon
ribbon_y0 = max_count_global * 1.06
ribbon_y1 = max_count_global * 1.13
# raise the ylim so labels + ribbon fit cleanly
y_top_full = max(y_top, ribbon_y1 + max_count_global * 0.10) + max_count_global * 0.01

for j in range(len(fine_x) - 1):
    sc = SCENARIOS[dominant_idx[j]]
    ax.fill_betweenx([ribbon_y0, ribbon_y1], fine_x[j], fine_x[j+1],
                      color=SCENARIO_COLOR[sc], alpha=0.85, linewidth=0)

ax.set_xlim(0, xmax_h)
ax.set_ylim(0, y_top_full)
ax.set_xlabel('Aggregated total global storage rate for 2050 (Gt/yr)')
ax.set_ylabel('Frequency')
ax.grid(axis='y', color='#eeeeee', lw=0.5)
ax.set_axisbelow(True)

# legends in the right margin
model_handles = [
    Line2D([0],[0], color='#444', ls='-',  lw=1.8, label='Logistic'),
    Line2D([0],[0], color='#444', ls='--', lw=1.8, label='Gompertz'),
]
leg1 = ax.legend(handles=model_handles,
                  bbox_to_anchor=(1.025, 1.0), loc='upper left',
                  frameon=False, title='Growth model', title_fontsize=9,
                  fontsize=9)
ax.add_artist(leg1)

# scenario legend with screened medians + reduction
scen_handles = []
for scen in SCENARIOS:
    s = summary[summary['scenario'] == scen]
    if s.empty: continue
    L = s[s['model']=='Logistic']
    G = s[s['model']=='Gompertz']
    mL = float(L['med_min'].values[0]) if not L.empty else np.nan
    mG = float(G['med_min'].values[0]) if not G.empty else np.nan
    scen_handles.append(
        Line2D([0],[0], color=SCENARIO_COLOR[scen], lw=2.4,
                label=f'{SCENARIO_LABEL[scen]:<10s}  L={mL:.1f}  G={mG:.1f}'))
ax.legend(handles=scen_handles,
           bbox_to_anchor=(1.025, 0.78), loc='upper left',
           frameon=False,
           title='Scenario  (post-screening median Gt/yr)',
           title_fontsize=9, fontsize=8.6)

fig.suptitle('Global aggregate 2050 storage rate — post-screening',
              fontsize=14, fontweight='bold', y=0.985)
plt.subplots_adjust(top=0.92, bottom=0.10, left=0.06, right=0.78)

png = FIG / 'fig_global_aggregate_2050_histograms_post_screening.png'
pdf = FIG / 'fig_global_aggregate_2050_histograms_post_screening.pdf'
fig.savefig(png, dpi=200, bbox_inches='tight')
fig.savefig(pdf, dpi=240, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {png}')
print(f'Saved: {pdf}')


In [ ]:
# 7. Mirror plot: pre vs post 2050 rate
# Two stacked subplots sharing the X-axis (2050 storage rate, Gt/yr).
# Top = pre-screening (frequency up), bottom = post-screening (down).
# Dominance ribbons sit on the outer edges of each half; median ticks
# in the gap show per-(scenario, model) median shifts.
from matplotlib.gridspec import GridSpec
from scipy.stats import gaussian_kde
from scipy.signal import medfilt

# (a) Re-build the unscreened bootstrap
rng_un = np.random.default_rng(SEED)
unscr_rows = []
for scen in SCENARIOS:
    p = P1 / f'samples_{scen}.csv'
    if not p.exists(): continue
    df = pd.read_csv(p)
    for mdl in MODELS:
        sub = df[df['Model'] == mdl]
        if sub.empty: continue
        country_arrays = {c: sub.loc[sub['Country']==c,'rate_2050'].to_numpy()
                           for c in sub['Country'].unique()}
        country_arrays = {c: a for c,a in country_arrays.items() if a.size > 0}
        if not country_arrays: continue
        world = np.zeros(N_BOOT)
        for c, arr in country_arrays.items():
            world += rng_un.choice(arr, size=N_BOOT, replace=True)
        unscr_rows.extend([{'scenario':scen,'model':mdl,'rate_2050_gt_yr':float(w)}
                            for w in world])
unscr_records = pd.DataFrame(unscr_rows)
scrn_records  = main_df.rename(columns={'screened':'rate_2050_gt_yr'})[
                   ['scenario','model','rate_2050_gt_yr']].copy()

xmax_m = float(np.percentile(unscr_records['rate_2050_gt_yr'], 99.5))

def dominance(records):
    fine_x = np.linspace(0.01, xmax_m, 300)
    dens = np.zeros((len(SCENARIOS), len(fine_x)))
    for i, scen in enumerate(SCENARIOS):
        v = records[records['scenario']==scen]['rate_2050_gt_yr'].to_numpy()
        if v.size > 1:
            dens[i] = gaussian_kde(v, bw_method=0.10)(fine_x)
    return fine_x, medfilt(np.argmax(dens, axis=0), kernel_size=11).astype(int)
def runs_of(idx, fine_x):
    out = []
    cur_i, cur_x0 = idx[0], fine_x[0]
    for j in range(1, len(fine_x)):
        if idx[j] != cur_i:
            out.append((cur_i, cur_x0, fine_x[j-1])); cur_i, cur_x0 = idx[j], fine_x[j]
    out.append((cur_i, cur_x0, fine_x[-1]))
    return out
fine_x_pre,  dom_pre  = dominance(unscr_records)
fine_x_post, dom_post = dominance(scrn_records)
runs_pre  = runs_of(dom_pre,  fine_x_pre)
runs_post = runs_of(dom_post, fine_x_post)

def hist_per_scenario_ax(records, ax, sign):
    max_count = 0
    for scen in SCENARIOS:
        col = SCENARIO_COLOR[scen]
        sub_scen = records[records['scenario'] == scen]
        if sub_scen.empty: continue
        s_lo = float(sub_scen['rate_2050_gt_yr'].min())
        s_hi = float(sub_scen['rate_2050_gt_yr'].max())
        pad  = max((s_hi - s_lo) * 0.04, 1e-3)
        lo, hi = max(0.0, s_lo - pad), s_hi + pad
        bins   = np.linspace(lo, hi, 33)
        bin_ctr = 0.5 * (bins[:-1] + bins[1:])
        for mdl in MODELS:
            v = sub_scen[sub_scen['model']==mdl]['rate_2050_gt_yr'].to_numpy()
            if v.size == 0: continue
            counts, _ = np.histogram(v, bins=bins)
            if counts.max() == 0: continue
            max_count = max(max_count, int(counts.max()))
            signed = sign * counts
            sty = MODEL_STYLE[mdl]
            if sty['fill_alpha'] > 0:
                ax.fill_between(bin_ctr, 0, signed, step='mid', color=col,
                                  alpha=sty['fill_alpha'], linewidth=0)
            ax.stairs(signed, bins, color=col, linestyle=sty['ls'],
                       linewidth=sty['lw'], alpha=0.92, baseline=None)
    return max_count

# (b) Build the figure: two stacked subplots, shared X
fig = plt.figure(figsize=(13.0, 9.8))
gs  = GridSpec(2, 1, height_ratios=[1, 1], hspace=0.18,
                top=0.92, bottom=0.07, left=0.08, right=0.84)
ax_top = fig.add_subplot(gs[0])
ax_bot = fig.add_subplot(gs[1], sharex=ax_top)

mx_pre  = hist_per_scenario_ax(unscr_records, ax_top, +1)
mx_post = hist_per_scenario_ax(scrn_records,  ax_bot, -1)

y_top = mx_pre  * 1.30
y_bot = -mx_post * 1.30
ax_top.set_ylim(0, y_top)
ax_bot.set_ylim(y_bot, 0)

# Dominance ribbons
TOP_R0, TOP_R1 = mx_pre * 1.05, mx_pre * 1.12
BOT_R0, BOT_R1 = -mx_post * 1.05, -mx_post * 1.12
LBL_TOP = [mx_pre * 1.16, mx_pre * 1.22]
LBL_BOT = [-mx_post * 1.16, -mx_post * 1.22]
for j in range(len(fine_x_pre) - 1):
    sc = SCENARIOS[dom_pre[j]]
    ax_top.fill_betweenx([TOP_R0, TOP_R1], fine_x_pre[j], fine_x_pre[j+1],
                          color=SCENARIO_COLOR[sc], alpha=0.85, linewidth=0)
for k, (sc_idx, x0, x1) in enumerate(runs_pre):
    if x1 - x0 < 0.10: continue
    sc = SCENARIOS[sc_idx]; xc = 0.5 * (x0 + x1)
    ax_top.text(xc, LBL_TOP[k % 2], SCENARIO_LABEL[sc],
                 ha='center', va='bottom', fontsize=8.4, fontweight='bold',
                 color=SCENARIO_COLOR[sc])
for j in range(len(fine_x_post) - 1):
    sc = SCENARIOS[dom_post[j]]
    ax_bot.fill_betweenx([BOT_R0, BOT_R1], fine_x_post[j], fine_x_post[j+1],
                          color=SCENARIO_COLOR[sc], alpha=0.85, linewidth=0)
for k, (sc_idx, x0, x1) in enumerate(runs_post):
    if x1 - x0 < 0.10: continue
    sc = SCENARIOS[sc_idx]; xc = 0.5 * (x0 + x1)
    ax_bot.text(xc, LBL_BOT[k % 2], SCENARIO_LABEL[sc],
                 ha='center', va='top', fontsize=8.4, fontweight='bold',
                 color=SCENARIO_COLOR[sc])

# Median ticks in the gap: solid L, dashed G
TICK_PRE_OUT  = mx_pre  * 0.08
TICK_POST_OUT = mx_post * 0.08
for scen in SCENARIOS:
    s = summary[summary['scenario']==scen]
    if s.empty: continue
    col = SCENARIO_COLOR[scen]
    for mdl in MODELS:
        sub = s[s['model']==mdl]
        if sub.empty: continue
        m_pre  = float(sub['med_un'].values[0])
        m_post = float(sub['med_min'].values[0])
        if mdl == 'Logistic':
            style, lwd = '-', 2.6
        else:
            style, lwd = (0, (3, 1.5)), 2.4
        ax_top.plot([m_pre, m_pre], [0, -TICK_PRE_OUT],
                     color=col, lw=lwd, linestyle=style, alpha=0.95,
                     zorder=15, clip_on=False, solid_capstyle='butt')
        ax_bot.plot([m_post, m_post], [0, TICK_POST_OUT],
                     color=col, lw=lwd, linestyle=style, alpha=0.95,
                     zorder=15, clip_on=False, solid_capstyle='butt')

# Y-axis tick labels (absolute frequency)
import math
def nice_ticks(maxv):
    if maxv <= 0: return []
    step = 10 ** math.floor(math.log10(maxv))
    if maxv / step < 2:   step *= 0.5
    elif maxv / step < 5: step *= 1
    else:                 step *= 2
    ticks = []; v = 0
    while v <= maxv: ticks.append(v); v += step
    return ticks
ax_top.set_yticks(nice_ticks(mx_pre))
ax_top.set_yticklabels([f'{int(t)}' for t in nice_ticks(mx_pre)])
ax_bot.set_yticks([-t for t in nice_ticks(mx_post) if t > 0] + [0])
ax_bot.set_yticklabels([f'{int(t)}' for t in nice_ticks(mx_post) if t > 0] + ['0'])

ax_top.set_ylabel('Frequency  (pre-screening)', fontsize=10)
ax_bot.set_ylabel('Frequency  (post-screening)', fontsize=10)
plt.setp(ax_top.get_xticklabels(), visible=False)
ax_top.tick_params(axis='x', length=0)
ax_bot.set_xlabel('Aggregated total global storage rate for 2050 (Gt/yr)')
ax_bot.set_xlim(0, xmax_m)
for axx in (ax_top, ax_bot):
    axx.grid(axis='y', color='#eeeeee', lw=0.5)
    axx.set_axisbelow(True)

model_handles = [
    Line2D([0],[0], color='#444', ls='-', lw=2.0, label='Logistic'),
    Line2D([0],[0], color='#444', ls=(0,(3,1.5)), lw=2.0, label='Gompertz'),
]
ax_top.legend(handles=model_handles,
                bbox_to_anchor=(1.03, 1.0), loc='upper left',
                frameon=False,
                title='Growth model', title_fontsize=9, fontsize=9,
                handlelength=2.4)

fig.suptitle('Global aggregate 2050 storage rate — pre vs post CO2BLOCK screening',
              fontsize=14, fontweight='bold', y=0.975)

png = FIG / 'fig_global_aggregate_2050_pre_post_mirror.png'
pdf = FIG / 'fig_global_aggregate_2050_pre_post_mirror.pdf'
fig.savefig(png, dpi=200, bbox_inches='tight')
fig.savefig(pdf, dpi=240, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {png}')
print(f'Saved: {pdf}')
